In [1]:

# Data handling
import pandas as pd
# Train-test split
from sklearn.model_selection import train_test_split
# Preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
# Pipeline
from sklearn.pipeline import Pipeline
# Model
from sklearn.tree import DecisionTreeClassifier
# Evaluation
from sklearn.metrics import accuracy_score, confusion_matrix


In [2]:
df = pd.read_csv(r"C:\Users\Shurs\Desktop\DATA SET\diabetes_prediction_dataset.csv")

In [3]:
df

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0
...,...,...,...,...,...,...,...,...,...
99995,Female,80.0,0,0,No Info,27.32,6.2,90,0
99996,Female,2.0,0,0,No Info,17.37,6.5,100,0
99997,Male,66.0,0,0,former,27.83,5.7,155,0
99998,Female,24.0,0,0,never,35.42,4.0,100,0


In [4]:
df.isnull().sum() # check missing values

gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64

In [5]:
df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  object 
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  object 
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 6.9+ MB


In [7]:
df.describe()

,age,hypertension,heart_disease,bmi,HbA1c_level,blood_glucose_level,diabetes
count,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,41.885856,0.07485,0.039420,27.320767,5.527507,138.058060,0.085000
std,22.516840,0.26315,0.194593,6.636783,1.070672,40.708136,0.278883
min,0.080000,0.00000,0.000000,10.010000,3.500000,80.000000,0.000000
25%,24.000000,0.00000,0.000000,23.630000,4.800000,100.000000,0.000000
50%,43.000000,0.00000,0.000000,27.320000,5.800000,140.000000,0.000000
75%,60.000000,0.00000,0.000000,29.580000,6.200000,159.000000,0.000000
max,80.000000,1.00000,1.000000,95.690000,9.000000,300.000000,1.000000


In [8]:
df['diabetes'].value_counts() # Data is inbalance non diabitic has much more in the data set and can affect accuracy scores

diabetes
0    91500
1     8500
Name: count, dtype: int64

In [9]:
X = df.drop("diabetes", axis = 1) # All collums except the target collumns i.e "diabetes"
y = df["diabetes"] # Target collumn 

In [10]:
print(X.columns)

Index(['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history',
       'bmi', 'HbA1c_level', 'blood_glucose_level'],
      dtype='object')


In [11]:
print(y.head())

0    0
1    0
2    0
3    0
4    0
Name: diabetes, dtype: int64


In [12]:
"diabetes" in X.columns # check for data leakage in X column

False

In [13]:
df["gender"].unique()

array(['Female', 'Male', 'Other'], dtype=object)

In [14]:
df["smoking_history"].unique()

array(['never', 'No Info', 'current', 'former', 'ever', 'not current'],
      dtype=object)

In [15]:
df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [16]:
cat = ["gender","smoking_history"]
num = ["age", "hypertension","heart_disease","bmi","HbA1c_level","blood_glucose_level"]

In [17]:
cat

['gender', 'smoking_history']

In [18]:
num

['age',
 'hypertension',
 'heart_disease',
 'bmi',
 'HbA1c_level',
 'blood_glucose_level']

In [19]:
"diabetes" in num

False

In [20]:
preprocessor  = ColumnTransformer([
    ("num",StandardScaler(),num),
    ("cat",OneHotEncoder(handle_unknown = "ignore"),cat)
])     
# use StandardScaler on numerical columns and OneHotEncoder on categorical columns using ColumnTransformer

In [21]:
# Fit pipeline for consistent data transformation during training and prediction.
pipeline = Pipeline([
    ("preprocessing",preprocessor),
    ("model",DecisionTreeClassifier(class_weight ="balanced",max_depth = 5))
])

In [22]:
# Split data for train and test keeping 20% of data for testing and ensuring same split evertime
X_train,X_test,y_train,y_test = train_test_split(
    X,y, test_size = 0.2, random_state = 42
)

In [23]:
pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'hypertension',
                                                   'heart_disease', 'bmi',
                                                   'HbA1c_level',
                                                   'blood_glucose_level']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['gender',
                                                   'smoking_history'])])),
                ('model',
                 DecisionTreeClassifier(class_weight='balanced', max_depth=5))])

In [24]:
y_pred = pipeline.predict(X_test) # prediction

In [25]:
accuracy = accuracy_score(y_test, y_pred) # Accuracy Score
print("Accuracy:", accuracy)

Accuracy: 0.8267


In [26]:
cm = confusion_matrix(y_test, y_pred) # Confusion Matrix
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[14909  3383]
 [   83  1625]]
